# Expert System (i.e. Knowledge-Based System) Shell/Engine - Educational DEMO
Author: Ryan J Urbanowicz, PhD

Institution: Cedars Sinai Health Sciences University

Date: 1/16/2026

Details: V4 - Extends backward chaining as a 'hybrid' option, also running forward chaining

This notebook includes code assembling a **basic but flexible expert system shell, built from the ground up** (i.e. without using existing expert system shells like CLIPS, clipspy, or PyKE), but certainly directly inspired by them. 

This expert system shell is primarily intended for educational purposes. While it is a realtively simple implementation it has been designed with a good deal of flexibility - not found in other Python or Java expert system implementations which are often confusing, not well documented, and or no longer supported/used often. This shell can be used to create a fairly wide varity of expert and knowledge-based system decision making/reasoning tools. Here is a summary of the key features of this shell impelemntation. 
1. Handles both **deductive** (certain) and **inductive** (uncertain/probabilistic) **reasoning**
2. Can perform both **forward** and **backward chaining** (i.e. reasoning)
3. Separates the knowledge base (saved as a json file) from the inference engine and other components (a simple knowlege base editor and a simple explanation system)
4. Uses an easy-to-understand and simple syntax for facts, rules, and questions in the knoweldge-base (however potentially limiting for some applications)
5. Flexibly handles truth comparisons including **(==, >, <, <=, >=,!=)** and fact-states such as True/False, yes/no, etc. 
6. For inductive reasoning - employs certainty factores (0-1 values) - rule firing propagates uncertainty with a product of cfs, while 'and's' of conjuntive rules take the minimum cfs across rule conditions. When multiple rules fire, certainty factors are combined with the Mycin cf update (i.e. new_cf = new_cf + old_cf * (1 - new_cf))
7. The code in the shell below has also been adapted into a 'Streamlit' web dashboard/GUI that can be easily shared and played with. (i.e. **'expert_system_app.py'**)
    * This dashboard is:
        * Run with the command: **'streamlit run expert_system_app.py'**
        * Limited to **backward chaining**.
        * Allows loading of different knowlege bases as .json files.
        * Requires selection of **deductive** vs. **probailistic** (i.e. inductive) reasoning before loading a knowelege base.
        * Supports selection of certainty factors for inputs using a slider.
        * Allows selection from available goals in knowledge base that can be proved.
        * Provides a breakdown of reasoning, and an explanation of decisions.

## Expert System Shell
This shell does not be edited in order to build an expert system, however since this shell is simply and concisely built from the ground up, there are many opportunities to improve and extend this implementation to the needs of the user. 

In [ ]:
import operator
import json

# Custom exception to break out of deep recursion/loops instantly
class GoalReachedException(Exception):
    def __init__(self, cf):
        self.cf = cf

class KnowledgeManager:
    @staticmethod
    def load_from_json(engine, file_path):
        with open(file_path, 'r') as f:
            data = json.load(f)

        for fact_name, text in data.get("questions", {}).items():
            engine.add_question(fact_name, text)

        for f_data in data.get("facts", []):
            try:
                engine.initialize_fact(f_data["name"], Fact(
                    f_data["name"], f_data["value"], f_data["cf"], f_data["explanation"]
                ))
            except:
                engine.initialize_fact(f_data["name"], Fact(
                    f_data["name"], f_data["value"], 1.0, f_data["explanation"]
                ))

        for r_data in data.get("rules", []):
            conditions = [Condition(c["fact1"], c["op"], c["fact2"]) for c in r_data["conditions"]]
            try:
                engine.add_rule(Rule(
                    r_data["id"], conditions, tuple(r_data["conclusion"]), 
                    r_data["cf"], r_data["explanation"]
                ))
            except:
                engine.add_rule(Rule(
                    r_data["id"], conditions, tuple(r_data["conclusion"]), 
                    1.0, r_data["explanation"]
                ))
        print(f"Successfully loaded {len(data['facts'])} facts, {len(data['rules'])} rules.")

# ... [Condition, Fact, Rule classes remain the same] ...
class Condition:
    def __init__(self, fact1, op, fact2):
        self.fact1, self.op, self.fact2 = fact1, op, fact2 
    def evaluate(self, f1_v, f2_v):
        return OPERATORS[self.op](f1_v, f2_v)

class Fact:
    def __init__(self, name, value, cf=1.0, explanation="Given", dependencies=None):
        self.name, self.value, self.cf, self.explanation = name, value, cf, explanation
        self.dependencies = dependencies if dependencies else []

class Rule:
    def __init__(self, id, conditions, conclusion, rule_cf, explanation):
        self.id, self.conditions, self.conclusion, self.rule_cf, self.explanation = id, conditions, conclusion, rule_cf, explanation

OPERATORS = {">": operator.gt, "<": operator.lt, ">=": operator.ge, "<=": operator.le, "==": operator.eq, "!=": operator.ne}

class ExpertSystem:
    def __init__(self, reasoning='deductive'):
        self.rules, self.facts, self.questions = [], {}, {}
        self.reasoning = reasoning
        self.target_goal = None  # Track the current top-level goal

    def add_fact(self, name, value, cf=1.0, explanation="Initial", dependencies=None):
        if name in self.facts: 
            old_cf = self.facts[name].cf
            self.facts[name].cf = old_cf + cf * (1 - old_cf)
            if dependencies: self.facts[name].dependencies.extend([d for d in dependencies if d not in self.facts[name].dependencies])
        else:
            self.facts[name] = Fact(name, value, cf, explanation, dependencies)
        
        # IMMEDIATELY STOP if the fact being added is our target goal
        if name == self.target_goal and self.facts[name].cf > 0:
            raise GoalReachedException(self.facts[name].cf)

    def add_rule(self, r): self.rules.append(r)
    def add_question(self, k, v): self.questions[k] = v
    def initialize_fact(self, n, f): self.facts[n] = f

    def get_explanation(self, fact_name):
        if fact_name not in self.facts:
            return "Fact ("+str(fact_name)+") not found."
        
        target_fact = self.facts[fact_name]
        lines = [f"Conclusion: {target_fact.name} is {target_fact.value} (Confidence: {target_fact.cf:.2%})"]
        lines.append("Reasoning Chain:")
        
        seen = set()
        
        def walk(name, level):
            if name in seen: return
            seen.add(name)
            f = self.facts.get(name)
            if not f: return
            indent = "  " * level
            marker = "└─" if level > 0 else "•"
            lines.append(f"{indent}{marker} {f.name}: {f.value} (CF={f.cf:.2f}) -- [{f.explanation}]")
            for dep in f.dependencies:
                walk(dep, level + 1)
        
        walk(fact_name, 0)
        return "\n".join(lines)
    
    # --- FORWARD CHAINING ---
    def forward_chain(self, verbose=False):
        fired_rules = set()
        changed = True
        while changed:
            changed = False
            for rule in self.rules:
                if rule.id in fired_rules: continue
                satisfied, min_cf, deps = True, 1.0, []
                for cond in rule.conditions:
                    f1 = self.facts.get(cond.fact1.removeprefix('$')) if cond.fact1.startswith('$') else Fact(cond.fact1, cond.fact1)
                    f2 = self.facts.get(cond.fact2.removeprefix('$')) if cond.fact2.startswith('$') else Fact(cond.fact2, cond.fact2)
                    if f1 and f2 and cond.evaluate(f1.value, f2.value):
                        min_cf = min(min_cf, f1.cf, f2.cf)
                        if cond.fact1.startswith('$'): deps.append(f1.name)
                        if cond.fact2.startswith('$'): deps.append(f2.name)
                    else: satisfied = False; break
                if satisfied:
                    # add_fact will raise GoalReachedException if conclusion == target_goal
                    self.add_fact(rule.conclusion[0], rule.conclusion[1], rule.rule_cf * min_cf, rule.explanation, deps)
                    fired_rules.add(rule.id); changed = True

    # --- BACKWARD CHAINING ---
    def backward_chain(self, goal_name):
        self.target_goal = goal_name
        print(f"\n--- Reasoning for: {goal_name} ---")
        if goal_name in self.facts: return self.facts[goal_name].cf
        try:
            return self._prove(goal_name)
        except GoalReachedException as e:
            print(f"Goal '{goal_name}' proved early. Terminating search.")
            return e.cf

    def _prove(self, goal_name):
        if goal_name in self.facts: return self.facts[goal_name].cf

        # 1. Attempt rule proof FIRST (Check without asking)
        rule_cfs = []
        for rule in self.rules:
            if rule.conclusion[0] == goal_name:
                satisfied, cond_cfs, deps = True, [], []
                for cond in rule.conditions:
                    c1_cf = self._prove(cond.fact1.removeprefix('$')) if cond.fact1.startswith('$') else 1.0
                    c2_cf = self._prove(cond.fact2.removeprefix('$')) if cond.fact2.startswith('$') else 1.0
                    f1_v = self.facts[cond.fact1.removeprefix('$')].value if cond.fact1.startswith('$') else cond.fact1
                    f2_v = self.facts[cond.fact2.removeprefix('$')].value if cond.fact2.startswith('$') else cond.fact2
                    
                    if min(c1_cf, c2_cf) > 0 and OPERATORS[cond.op](f1_v, f2_v):
                        cond_cfs.append(min(c1_cf, c2_cf))
                        if cond.fact1.startswith('$'): deps.append(cond.fact1.removeprefix('$'))
                        if cond.fact2.startswith('$'): deps.append(cond.fact2.removeprefix('$'))
                    else: satisfied = False; break
                
                if satisfied:
                    new_cf = rule.rule_cf * min(cond_cfs)
                    rule_cfs.append(new_cf)
                    # add_fact here might trigger GoalReachedException
                    self.add_fact(goal_name, rule.conclusion[1], new_cf, rule.explanation, deps)

        if rule_cfs:
            res = rule_cfs[0]
            for c in rule_cfs[1:]: res = res + c * (1 - res)
            return res

        # 2. Only if rules failed, ask User
        if goal_name in self.questions:
            print(f"QUERY: {self.questions[goal_name]}")
            val_in = input("Value: ")
            try: val = float(val_in)
            except: val = val_in.lower().strip()
            conf = 1.0 if self.reasoning == 'deductive' else float(input("Confidence (0-1): "))
            
            self.add_fact(goal_name, val, conf, "User provided")
            # 3. After answer, run recursive Forward Chain (Inference Cycle)
            self.forward_chain()
            return self.facts[goal_name].cf
            
        return 0.0


In [ ]:
import operator
import json

class KnowledgeManager:
    @staticmethod
    def load_from_json(engine, file_path):
        with open(file_path, 'r') as f:
            data = json.load(f)

        # 1. Load Questions
        for fact_name, text in data.get("questions", {}).items():
            engine.add_question(fact_name, text)

        # 2. Load Facts
        for f_data in data.get("facts", []):
            try:
                engine.initialize_fact(f_data["name"], Fact(
                    f_data["name"],
                    f_data["value"],
                    f_data["cf"],
                    f_data["explanation"],
                    None 
                ))
            except:
                engine.initialize_fact(f_data["name"], Fact(
                    f_data["name"],
                    f_data["value"],
                    1.0,
                    f_data["explanation"],
                    None
                ))

        # 3. Load Rules
        for r_data in data.get("rules", []):
            conditions = []
            for c_data in r_data["conditions"]:
                conditions.append(Condition(
                    c_data["fact1"], 
                    c_data["op"], 
                    c_data["fact2"],
                ))
            try:
                engine.add_rule(Rule(
                    r_data["id"],
                    conditions,
                    tuple(r_data["conclusion"]),
                    r_data["cf"],
                    r_data["explanation"]
                ))
            except:
                engine.add_rule(Rule(
                    r_data["id"],
                    conditions,
                    tuple(r_data["conclusion"]),
                    1.0,
                    r_data["explanation"]
                ))
        print(f"Successfully loaded {len(data['facts'])} facts, {len(data['rules'])} rules and {len(data['questions'])} questions.")

    @staticmethod
    def save_to_json(engine, file_path):
        data = {
            "questions": engine.questions,
            "facts": [
                {
                    "name": f.name,
                    "value": f.value,
                    "cf": f.cf,
                    "explanation": f.explanation
                } for f in engine.facts.values()
            ],
            "rules": [
                {
                    "id": r.id,
                    "conditions": [vars(c) for c in r.conditions],
                    "conclusion": r.conclusion,
                    "cf": r.rule_cf,
                    "explanation": r.explanation
                } for r in engine.rules
            ]
        }
        with open(file_path, 'w') as f:
            json.dump(data, f, indent=2)


# Define supported operators for numerical comparison
OPERATORS = {
    ">": operator.gt,
    "<": operator.lt,
    ">=": operator.ge,
    "<=": operator.le,
    "==": operator.eq,
    "!=": operator.ne
}

class Condition:
    def __init__(self, fact1, op, fact2):
        self.fact1 = fact1 
        self.op = op 
        self.fact2 = fact2 

    def evaluate(self,fact1_value,fact2_value):
        result = OPERATORS[self.op](fact1_value, fact2_value)
        return result


class Fact:
    def __init__(self, name, value, cf=1.0, explanation="Given as initial fact", dependencies=None):
        self.name = name
        self.value = value
        self.cf = cf
        self.explanation = explanation
        self.dependencies = dependencies if dependencies else []

    def __repr__(self):
        return f"Fact({self.name}={self.value}, CF={self.cf:.2f})"


class Rule:
    def __init__(self, id, conditions, conclusion, rule_cf, explanation):
        self.id = id
        self.conditions = conditions 
        self.conclusion = conclusion 
        self.rule_cf = rule_cf
        self.explanation = explanation


class ExpertSystem:
    def __init__(self, reasoning='deductive'):
        self.rules = []
        self.facts = {} 
        self.questions = {} 
        self.reasoning = reasoning

    def clear_facts(self):
        self.facts = {}
        
    def add_rule(self, rule):
        self.rules.append(rule)

    def initialize_fact(self, name, fact):
        self.facts[name] = fact

    def add_fact(self, name, value, cf=1.0, explanation="Initial", dependencies=None):
        new_fact = Fact(name, value, cf, explanation, dependencies)
        if name in self.facts: 
            old_cf = self.facts[name].cf
            self.facts[name].cf = old_cf + cf * (1 - old_cf)
            if dependencies:
                # Use set to avoid duplicate dependency listing if rules re-fire
                existing_deps = set(self.facts[name].dependencies)
                for d in dependencies:
                    if d not in existing_deps:
                        self.facts[name].dependencies.append(d)
        else:
            self.facts[name] = new_fact

    def add_question(self, fact_name, text):
        self.questions[fact_name] = text

    def get_explanation(self, fact_name):
        if fact_name not in self.facts:
            return "Fact ("+str(fact_name)+") not found."
        
        target_fact = self.facts[fact_name]
        lines = [f"Conclusion: {target_fact.name} is {target_fact.value} (Confidence: {target_fact.cf:.2%})"]
        lines.append("Reasoning Chain:")
        
        seen = set()
        
        def walk(name, level):
            if name in seen: return
            seen.add(name)
            f = self.facts.get(name)
            if not f: return
            indent = "  " * level
            marker = "└─" if level > 0 else "•"
            lines.append(f"{indent}{marker} {f.name}: {f.value} (CF={f.cf:.2f}) -- [{f.explanation}]")
            for dep in f.dependencies:
                walk(dep, level + 1)
        
        walk(fact_name, 0)
        return "\n".join(lines)

    # --- FORWARD CHAINING (Used recursively within Backward Chaining) ---
    def forward_chain(self, verbose=True):
        if verbose: print("\n--- Running Inference Cycle ---")
        fired_rules = set()
        changed = True
        while changed:
            changed = False
            for rule in self.rules:
                if rule.id in fired_rules:
                    continue

                satisfied = True
                min_cf = 1.0
                current_deps = [] 
                
                for cond in rule.conditions:
                    # Logic to resolve $ variables vs hardcoded values
                    if cond.fact1.startswith('$') and cond.fact2.startswith('$'):
                        fact1 = self.facts.get(cond.fact1.removeprefix('$'))
                        fact2 = self.facts.get(cond.fact2.removeprefix('$'))
                    elif cond.fact1.startswith('$'):
                        fact1 = self.facts.get(cond.fact1.removeprefix('$'))
                        self.add_fact(cond.fact2, cond.fact2, 1.0, 'Built-in')
                        fact2 = self.facts.get(cond.fact2)
                    elif cond.fact2.startswith('$'):
                        self.add_fact(cond.fact1, cond.fact1, 1.0, 'Built-in')
                        fact1 = self.facts.get(cond.fact1)
                        fact2 = self.facts.get(cond.fact2.removeprefix('$'))
                    else:
                        satisfied = False; break

                    if fact1 and fact2 and cond.evaluate(fact1.value, fact2.value):
                        min_cf = min(min_cf, fact1.cf, fact2.cf)
                        if cond.fact1.startswith('$'): current_deps.append(fact1.name)
                        if cond.fact2.startswith('$'): current_deps.append(fact2.name)
                    else:
                        satisfied = False; break

                if satisfied:
                    name, val = rule.conclusion
                    new_cf = rule.rule_cf * min_cf
                    self.add_fact(name, val, new_cf, rule.explanation, dependencies=list(set(current_deps)))
                    if verbose: print(f"  Rule Fired: {rule.id} -> {name} is {val}")
                    fired_rules.add(rule.id)
                    changed = True

    # --- BACKWARD CHAINING ---
    def backward_chain(self, goal_name):
        print(f"\n--- Backward Chaining for Goal: {goal_name} ---")
        return self._prove(goal_name)
    
    def _prove(self, goal_name):
        # If we already know it, return CF
        if goal_name in self.facts: return self.facts[goal_name].cf

        # Try to prove via rules
        rule_cfs = []
        for rule in self.rules:
            if rule.conclusion[0] == goal_name:
                rule_satisfied = True
                cond_cfs = [] 
                
                for cond in rule.conditions:
                    res_cf = [1.0] 
                    # Recursive calls to prove prerequisites
                    if cond.fact1.startswith('$'):
                        res_cf.append(self._prove(cond.fact1.removeprefix('$')))
                    if cond.fact2.startswith('$'):
                        res_cf.append(self._prove(cond.fact2.removeprefix('$')))
                    
                    # Fetch resolved facts
                    f1_name = cond.fact1.removeprefix('$') if cond.fact1.startswith('$') else cond.fact1
                    f2_name = cond.fact2.removeprefix('$') if cond.fact2.startswith('$') else cond.fact2
                    
                    # Ensure constants exist in fact base for comparison
                    if not cond.fact1.startswith('$'): self.add_fact(f1_name, f1_name, 1.0, "Literal")
                    if not cond.fact2.startswith('$'): self.add_fact(f2_name, f2_name, 1.0, "Literal")
                    
                    fact1 = self.facts.get(f1_name)
                    fact2 = self.facts.get(f2_name)

                    if min(res_cf) > 0 and fact1 and fact2 and cond.evaluate(fact1.value, fact2.value):
                        cond_cfs.append(min(res_cf))
                    else:
                        rule_satisfied = False; break                  
                
                if rule_satisfied:
                    new_cf = rule.rule_cf * min(cond_cfs)
                    rule_cfs.append(new_cf)
                    # We don't add the fact here manually because forward_chain will catch it
                    # however, we must ensure it's in the rule_cfs for MYCIN calc below.

        if rule_cfs:
            res = rule_cfs[0]
            for c in rule_cfs[1:]: res = res + c * (1 - res)
            return res

        # If rules failed, ask the user
        if goal_name in self.questions:
            print(f"QUERY: {self.questions[goal_name]}")
            val_input = input("Value: ")
            try: val = float(val_input)
            except: val = val_input.lower().strip()
            
            conf = 1.0 if self.reasoning == 'deductive' else float(input("Confidence (0.0-1.0): "))
            
            # Add user fact
            self.add_fact(goal_name, val, conf, "User provided")
            
            # NEW: Recursively check all rules after obtaining new input
            # This triggers forward chaining to find any newly possible conclusions
            self.forward_chain(verbose=True)
            
            # Re-check if the forward chain populated our current goal
            if goal_name in self.facts:
                return self.facts[goal_name].cf
                
            return conf
            
        return 0.0

In [ ]:
import operator
import json

class KnowledgeManager:
    @staticmethod
    def load_from_json(engine, file_path):
        with open(file_path, 'r') as f:
            data = json.load(f)

        # 1. Load Questions
        for fact_name, text in data.get("questions", {}).items():
            engine.add_question(fact_name, text)

        # 2. Load Facts
        for f_data in data.get("facts", []):
            try:
                engine.initialize_fact(f_data["name"], Fact(
                    f_data["name"],
                    f_data["value"],
                    f_data["cf"],
                    f_data["explanation"],
                    None 
                ))
            except:
                engine.initialize_fact(f_data["name"], Fact(
                    f_data["name"],
                    f_data["value"],
                    1.0,
                    f_data["explanation"],
                    None
                ))

        # 3. Load Rules
        for r_data in data.get("rules", []):
            conditions = []
            for c_data in r_data["conditions"]:
                conditions.append(Condition(
                    c_data["fact1"], 
                    c_data["op"], 
                    c_data["fact2"],
                ))
            try:
                engine.add_rule(Rule(
                    r_data["id"],
                    conditions,
                    tuple(r_data["conclusion"]),
                    r_data["cf"],
                    r_data["explanation"]
                ))
            except:
                engine.add_rule(Rule(
                    r_data["id"],
                    conditions,
                    tuple(r_data["conclusion"]),
                    1.0,
                    r_data["explanation"]
                ))
        print(f"Successfully loaded {len(data['facts'])} facts, {len(data['rules'])} rules and {len(data['questions'])} questions.")

    @staticmethod
    def save_to_json(engine, file_path):
        data = {
            "questions": engine.questions,
            "facts": [
                {
                    "name": f.name,
                    "value": f.value,
                    "cf": f.cf,
                    "explanation": f.explanation
                } for f in engine.facts.values()
            ],
            "rules": [
                {
                    "id": r.id,
                    "conditions": [vars(c) for c in r.conditions],
                    "conclusion": r.conclusion,
                    "cf": r.rule_cf,
                    "explanation": r.explanation
                } for r in engine.rules
            ]
        }
        with open(file_path, 'w') as f:
            json.dump(data, f, indent=2)


# Define supported operators for numerical comparison
OPERATORS = {
    ">": operator.gt,
    "<": operator.lt,
    ">=": operator.ge,
    "<=": operator.le,
    "==": operator.eq,
    "!=": operator.ne
}

class Condition:
    def __init__(self, fact1, op, fact2):
        self.fact1 = fact1 
        self.op = op 
        self.fact2 = fact2 

    def evaluate(self,fact1_value,fact2_value):
        result = OPERATORS[self.op](fact1_value, fact2_value)
        return result


class Fact:
    def __init__(self, name, value, cf=1.0, explanation="Given as initial fact", dependencies=None):
        self.name = name
        self.value = value
        self.cf = cf
        self.explanation = explanation
        self.dependencies = dependencies if dependencies else []

    def __repr__(self):
        return f"Fact({self.name}={self.value}, CF={self.cf:.2f})"


class Rule:
    def __init__(self, id, conditions, conclusion, rule_cf, explanation):
        self.id = id
        self.conditions = conditions 
        self.conclusion = conclusion 
        self.rule_cf = rule_cf
        self.explanation = explanation


class ExpertSystem:
    def __init__(self, reasoning='deductive'):
        self.rules = []
        self.facts = {} 
        self.questions = {} 
        self.reasoning = reasoning

    def clear_facts(self):
        self.facts = {}
        
    def add_rule(self, rule):
        self.rules.append(rule)

    def initialize_fact(self, name, fact):
        self.facts[name] = fact

    def add_fact(self, name, value, cf=1.0, explanation="Initial", dependencies=None):
        new_fact = Fact(name, value, cf, explanation, dependencies)
        if name in self.facts: 
            old_cf = self.facts[name].cf
            self.facts[name].cf = old_cf + cf * (1 - old_cf) #  handles the MYCIN accumulation logic automatically.
            if dependencies:
                self.facts[name].dependencies.extend(dependencies)
        else:
            self.facts[name] = new_fact

    def add_question(self, fact_name, text):
        self.questions[fact_name] = text

    # --- (2) CERTAINTY & (5) EXPLANATION SYSTEM ---
    def get_explanation(self, fact_name):
        if fact_name not in self.facts:
            return "Fact ("+str(fact_name)+") not found."
        
        target_fact = self.facts[fact_name]
        
        # Build the trace string
        lines = [f"Conclusion: {target_fact.name} is {target_fact.value} (Confidence: {target_fact.cf:.2%})"]
        lines.append("Reasoning Chain:")
        
        seen = set()
        
        def walk(name, level):
            if name in seen: return
            seen.add(name)
            
            f = self.facts.get(name)
            if not f: return
            
            indent = "  " * level
            marker = "└─" if level > 0 else "•"
            
            lines.append(f"{indent}{marker} {f.name}: {f.value} (CF={f.cf:.2f}) -- [{f.explanation}]")
            
            for dep in f.dependencies:
                walk(dep, level + 1)
        
        walk(fact_name, 0)
        return "\n".join(lines)

    # --- (3) FORWARD CHAINING ---
    def forward_chain(self, verbose=True):
        if verbose: print("\n--- Forward Chaining Inference Cycle ---")
        
        # Track which rules have fired to prevent infinite loops (double-counting same evidence)
        # while ensuring we cycle until all *other* possible rules fire.
        fired_rules = set()
        
        changed = True
        while changed:
            changed = False
            for rule in self.rules:
                # Skip rules that have already executed once
                if rule.id in fired_rules:
                    continue

                satisfied = True
                min_cf = 1.0
                current_deps = [] 
                
                for cond in rule.conditions:
                    if cond.fact1.startswith('$') and cond.fact2.startswith('$'):
                        fact1 = self.facts.get(cond.fact1.removeprefix('$'))
                        fact2 = self.facts.get(cond.fact2.removeprefix('$'))
                        if fact1: current_deps.append(fact1.name)
                        if fact2: current_deps.append(fact2.name)
                    elif cond.fact1.startswith('$') and not cond.fact2.startswith('$'):
                        fact1 = self.facts.get(cond.fact1.removeprefix('$'))
                        self.add_fact(cond.fact2, cond.fact2, 1.0, 'Built-in fact')
                        fact2 = self.facts.get(cond.fact2)
                        if fact1: current_deps.append(fact1.name)
                    elif not cond.fact1.startswith('$') and cond.fact2.startswith('$'):
                        self.add_fact(cond.fact1, cond.fact1, 1.0, 'Built-in fact')
                        fact1 = self.facts.get(cond.fact1)
                        fact2 = self.facts.get(cond.fact2.removeprefix('$'))
                        if fact2: current_deps.append(fact2.name)
                    else:
                        print("ERROR:Rules cannot be defined based on two hard-coded facts")

                    if fact1 and fact2 and cond.evaluate(fact1.value,fact2.value):
                        min_cf = min(min_cf, fact1.cf, fact2.cf)
                    else:
                        satisfied = False; break

                if satisfied:
                    name, val = rule.conclusion
                    new_cf = rule.rule_cf * min_cf
                    
                    # UPDATED: Always fire the rule if satisfied and not yet fired.
                    
                    self.add_fact(name, val, new_cf, rule.explanation, dependencies=current_deps)
                    
                    if verbose: print(f"  Rule Fired: {rule.id} -> {name} is {val} with cf {new_cf:.2%}")
                    
                    fired_rules.add(rule.id) # Mark rule as fired
                    changed = True

    # --- (3) & (4) BACKWARD CHAINING WITH QUESTION LOOP ---
    def backward_chain(self, goal_name, hybrid=False):
        print(f"\n--- Backward Chaining for Goal: {goal_name} ---")
        return self._prove(goal_name,hybrid)
    
    def hybrid_chain(self, goal_name, hybrid=True):
        print(f"\n--- Hybrid Chaining for Goal: {goal_name} ---")
        return self._prove(goal_name,hybrid)
    
    def _prove(self, goal_name, hybrid):
        if goal_name in self.facts: return self.facts[goal_name].cf

        rule_cfs = []
        for rule in self.rules:
            if rule.conclusion[0] == goal_name:
                rule_satisfied = True
                cond_cfs = [] 
                current_deps = [] 
                
                for cond in rule.conditions:
                    res_cf = [1.0] 
                    if cond.fact1.startswith('$') and cond.fact2.startswith('$'):
                        n1 = cond.fact1.removeprefix('$')
                        n2 = cond.fact2.removeprefix('$')
                        res_cf.append(self._prove(n1, hybrid)) 
                        fact1 = self.facts.get(n1)
                        res_cf.append(self._prove(n2, hybrid)) 
                        fact2 = self.facts.get(n2)
                        if fact1: current_deps.append(n1)
                        if fact2: current_deps.append(n2)
                    elif cond.fact1.startswith('$') and not cond.fact2.startswith('$'):
                        n1 = cond.fact1.removeprefix('$')
                        res_cf.append(self._prove(n1, hybrid)) 
                        fact1 = self.facts.get(n1)
                        res_cf.append(1.0)
                        self.add_fact(cond.fact2, cond.fact2, 1.0, 'Built-in fact')
                        fact2 = self.facts.get(cond.fact2)
                        if fact1: current_deps.append(n1)
                    elif not cond.fact1.startswith('$') and cond.fact2.startswith('$'):
                        n2 = cond.fact2.removeprefix('$')
                        res_cf.append(1.0)
                        self.add_fact(cond.fact1, cond.fact1, 1.0, 'Built-in fact')
                        fact1 = self.facts.get(cond.fact1)
                        res_cf.append(self._prove(n2, hybrid)) 
                        fact2 = self.facts.get(n2)
                        if fact2: current_deps.append(n2)
                    else:
                        print("ERROR:Rules cannot be defined based on two hard-coded facts")

                    if min(res_cf) > 0 and fact1 and fact2 and cond.evaluate(fact1.value,fact2.value):
                        cond_cfs.append(min(res_cf)) 
                    else:
                        rule_satisfied = False; break                  
                
                if rule_satisfied:
                    min_cond_cfs = min(cond_cfs) 
                    new_cf = rule.rule_cf * min_cond_cfs 
                    rule_cfs.append(new_cf)
                    self.add_fact(rule.conclusion[0], rule.conclusion[1], new_cf, rule.explanation, dependencies=current_deps)

        if rule_cfs:
            res = rule_cfs[0]
            for c in rule_cfs[1:]: res = res + c * (1 - res)
            return res

        # 2. Only if rules failed, ask User
        if goal_name in self.questions:
            print(f"QUERY: {self.questions[goal_name]}")
            val_in = input("Value: (e.g., 38.5 or 'yes'): ")
            try: val = float(val_in)
            except: val = val_in.lower().strip()
            conf = 1.0 if self.reasoning == 'deductive' else float(input("Confidence (0.0 to 1.0): "))
            
            self.add_fact(goal_name, val, conf, "User provided")
            if hybrid:
                # 3. After answer, run recursive Forward Chain (Inference Cycle)
                self.forward_chain(verbose=True)
            else:
                pass
            return self.facts[goal_name].cf
            
        return 0.0


## Introduction to Knowledge Base Syntax
Below we explore simple expert systems designed as accessible examples.
* Each knowlege base is stored within a .json file. 
* The .json knowlege base syntax is that of a dictionary organized by 'questions', 'facts' and 'rules'. Each type of entry has its own required syntax.
    * 'questions' - a dictionary of key:value pairs where the key is the name of a fact needed by the system and the value is the text description of the input needed seen by the user.
    * 'facts' - an array of objects (i.e. a list of dictionaries) where each fact has the keys (name, value, cf, explanation) identifying the fact name, it's assigned value, it's certainty factor, and a description of the fact (used by the explanation system)
    * 'rules' - an array of objects (i.e. a list of dictionaries) where each rule has the keys (id, conditions, conclusion, cf, explanation) 
        * 'Conditions' are list objects that can include one or more conditions (to be satisfied) in order for the rule to 'fire' (i.e. add the conclusion as a new fact in the knowledge base)
        * Each condition has two 'facts' that are compared using any of the following (==, >, <, <=, >=,!=). 
        * These 'facts' can either be variables (starting with '$') (which can also be defined as facts in the KB directly) or static (hard-coded) values within the condition itself, that the system will automatically turn into 'implied facts'. 
        * 'Conclusions' are given as a tuple that first defines the [fact name, fact value] when the rule fires.
        * 'cf' and 'explanation' mean the same as for 'facts'

### Weather Knowlege Base (Example) - Deductive Reasoning - Forward Chaining
This is a simple expert system designed to provide users guidance on 'what to bring' when leaving the house based on current weather conditions.
* This knowledge base can be found in 'kb_weather.json'.
* In this forward chaining example, only facts and rules are used (not questions).
* Neither facts, rules, nor questions include certainty factors in this knowledge base, so the system will assume they all have a certainty factor of 1.0 by default.

The code below applies 'forward chaining' reasoning to this simple deductive reasoning example knowledge base.


In [2]:
# 1. Initialize the empty engine
engine = ExpertSystem('deductive') #deductive or probabilistic

# 2. Load the Knowledge (Assuming you saved the JSON above as 'kb.json')
# For this demo, we'll simulate the file loading
try:
    KnowledgeManager.load_from_json(engine, 'kb_weather.json')
except FileNotFoundError:
    print("Error: Knowledge base file not found.")

# Manually add some facts for testing forward chaining (i.e. part of the simple knowlege base editor)
engine.add_fact("precipitation", "yes")
engine.add_fact("windy", "yes") 
engine.add_fact("temperature", 80)

# Specify the 'goal' (i.e. target fact) we want to determine from the knowlege base
goal_name = "what to bring"

# 3. Use the system for a decision making
print("\n--- AUTOMATED DECISION MAKING ---")

# Forward chaining will now use the rules and facts loaded from JSON
engine.forward_chain()

# 4. Display Explained Results
print("\n" + "="*40)
print(engine.get_explanation(goal_name))
print("="*30)

#print(engine.facts) #optionally print all facts stored in the knowlege base at the end of forward chaining
engine.clear_facts() #cleanup expert system fact memory for another run

Successfully loaded 3 facts, 9 rules.

--- AUTOMATED DECISION MAKING ---

Conclusion: what to bring is raincoat (Confidence: 100.00%)
Reasoning Chain:
• what to bring: raincoat (CF=1.00) -- [Bring raincoat when it's raining and windy.]
  └─ rain_protect: True (CF=1.00) -- [Rain protection needed when it's raining.]
    └─ precipitation: yes (CF=1.00) -- [Initial]
    └─ temperature: 80 (CF=1.00) -- [Initial]
    └─ freeze_threshold: 0.0 (CF=1.00) -- [Threshold for water freezing (in Fahrenheit).]
  └─ windy: yes (CF=1.00) -- [Initial]


AttributeError: 'ExpertSystem' object has no attribute 'clear_facts'

### Weather Knowlege Base (Example) - Deductive Reasoning - Backward Chaining
The code below applies 'backward chaining' reasoning to this simple deductive reasoning example knowledge base. When run, a prompt will appear asking for user inputs generated by the questions built into the knowlege base. This example will use everything in the knowledge base, i.e. facts, rules, and questions.

In [ ]:
# 1. Initialize the empty engine
engine = ExpertSystem('deductive')

# 2. Load the Knowledge (Assuming you saved the JSON above as 'kb.json')
# For this demo, we'll simulate the file loading
try:
    KnowledgeManager.load_from_json(engine, 'kb_weather.json')
except FileNotFoundError:
    print("Error: Knowledge base file not found.")

# Specify the 'goal' (i.e. target fact) we want to determine from the knowlege base
goal_name = "what to bring"

# 3. Use the system for a diagnosis
print("\n--- MODULAR DIAGNOSTIC SESSION ---")
# Backward chaining will now use the rules and questions loaded from JSON
engine.backward_chain(goal_name)

# 4. Display Explained Results
print("\n" + "="*40)
print(engine.get_explanation(goal_name))
print("="*30)

## Simple Diagnosis Knowlege Base (Example) - Inductive Reasoning - Forward Chaining
This is a simple expert system designed to provide users guidance on 'diagnosis' of flu.
* This knowledge base can be found in 'kb_diagnose_flu.json'.
* In this forward chaining example, only facts and rules are used (not questions).
* Facts, and rules include certainty factors in this knowledge base denoted by 'cf'.
* Users will input certainty factors along with any input facts added at the start of forward chaining

The code below applies 'forward chaining' reasoning to this simple inductive reasoning example knowledge base.

In [ ]:
# 1. Initialize the empty engine
engine = ExpertSystem('probabilistic') #deductive or probabilistic

# 2. Load the Knowledge (Assuming you saved the JSON above as 'kb.json')
# For this demo, we'll simulate the file loading
try:
    KnowledgeManager.load_from_json(engine, 'kb_diagnose_flu.json')
except FileNotFoundError:
    print("Error: Knowledge base file not found.")

# Manually add some facts for testing forward chaining (i.e. part of the simple knowlege base editor)
engine.add_fact("temperature", 101, 1.0)
engine.add_fact("cough", "yes", 1.0) 

# Specify the 'goal' (i.e. target fact) we want to determine from the knowlege base
goal_name = "diagnosis"

# 3. Use the system for a decision making
print("\n--- AUTOMATED DECISION MAKING ---")

# Forward chaining will now use the rules and facts loaded from JSON
engine.forward_chain()

# 4. Display Explained Results
print("\n" + "="*40)
print(engine.get_explanation(goal_name))
print("="*30)

#print(engine.facts) #optionally print all facts stored in the knowlege base at the end of forward chaining
engine.clear_facts() #cleanup expert system fact memory for another run

## Simple Diagnosis Knowlege Base (Example) - Inductive Reasoning - Backward Chaining
The code below applies 'backward chaining' reasoning to this simple deductive reasoning example knowledge base. When run, a prompt will appear asking for user inputs generated by the questions built into the knowlege base. This example will use everything in the knowledge base, i.e. facts, rules, and questions.
* Users will input certainty factors (cf) along with any input questions asked at the start of backward chaining

The code below applies 'backward chaining' reasoning to this simple inductive reasoning example knowledge base.

In [ ]:
# 1. Initialize the empty engine
engine = ExpertSystem('probabilistic')

# 2. Load the Knowledge (Assuming you saved the JSON above as 'kb.json')
# For this demo, we'll simulate the file loading
try:
    KnowledgeManager.load_from_json(engine, 'kb_diagnose_flu.json')
except FileNotFoundError:
    print("Error: Knowledge base file not found.")

# Specify the 'goal' (i.e. target fact) we want to determine from the knowlege base
goal_name = "diagnosis"

# 3. Use the system for a diagnosis
print("\n--- MODULAR DIAGNOSTIC SESSION ---")
# Backward chaining will now use the rules and questions loaded from JSON
engine.backward_chain(goal_name)

# 4. Display Explained Results
print("\n" + "="*40)
print(engine.get_explanation(goal_name))
print("="*30)